In [38]:
import pandas as pd 
import numpy as np
import tensorflow as tf
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
import matplotlib.pyplot as plt


In [39]:
data = pd.read_csv("/Users/joulesss/Documents/DOCTORADO/Materias/AnaliticaComputacional/Proyecto2_Final/data_encoded.csv")

data.head(5)


,periodo,cole_area_ubicacion,cole_bilingue,cole_caracter,cole_cod_mcpio_ubicacion,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,estu_genero,...,fami_personashogar,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20162,1,0,2,15238,1,DUITAMA,1,COLEGIO GUILLERMO LEON VALENCIA,1.0,...,NaN,NaN,NaN,NaN,61.0,59.0,63.0,57.0,56.0,295.0
1,20142,0,0,2,15759,1,SOGAMOSO,1,INSTITUCION EDUCATIVA LA INDEPENDENCIA,0.0,...,6.0,1.0,0.0,0.0,52.0,55.0,54.0,60.0,54.0,277.0
2,20152,1,0,1,15835,6,TURMEQUE,1,I.E. TECNICA INDUSTRIAL,0.0,...,3.0,0.0,0.0,1.0,55.0,52.0,50.0,53.0,47.0,254.0
3,20172,1,0,2,15176,1,CHIQUINQUIRA,1,I.E. TECNICA PIO ALBERTO FERRO PEÑA,0.0,...,5.5,0.0,0.0,1.0,59.0,58.0,46.0,51.0,45.0,253.0
4,20172,1,0,2,15759,1,SOGAMOSO,0,COLEGIO COOPERATIVO REYES PATRIA,1.0,...,3.5,1.0,1.0,1.0,75.0,66.0,70.0,65.0,64.0,335.0


## Trtamiento de variables

In [40]:

data['nivel_educacion_padres'] = round((data['fami_educacionmadre']+data['fami_educacionpadre'])/2,0)# tomar el máximo nivel entre ambos padres

data['hacinamiento'] = data['fami_personashogar'] / (data['fami_cuartoshogar'] + 1)


In [41]:
""" def clasificar_ingles(x):
    if 0 <= x <= 36:
        return 0   # Pre A1
    elif 37 <= x <= 57:
        return 1   # A1
    elif 58 <= x <= 70:
        return 2   # A2
    elif 71 <= x <= 100:
        return 3   # B1
    else:
        return None """
        
def clasificar_ingles(x):
    if x <= 50:
        return 0
    else: 
        return 1 
                
 
data["nivel_ingles"] = data["punt_ingles"].apply(clasificar_ingles)

## Variable uso de modelo 

In [42]:
var = ["cole_bilingue", "cole_jornada","cole_area_ubicacion","cole_naturaleza", "cole_caracter","cole_area_ubicacion", "cole_mcpio_ubicacion",
       "fami_tienecomputador","fami_tieneinternet","fami_estratovivienda", "nivel_educacion_padres","hacinamiento","nivel_ingles"]


data_p1 = data[var].copy().dropna()


y = data_p1["nivel_ingles"] #var interes 

municipio = data_p1["cole_mcpio_ubicacion"]

X = data_p1.drop(columns=["nivel_ingles", "cole_mcpio_ubicacion"])

### Partición de datos

In [43]:
# 70% Train - 30% temp -> redirigido a test y validacion 
X_train, X_temp, y_train, y_temp, municipio_train, municipio_temp = train_test_split(
    X, y, municipio,
    test_size=0.30,
    random_state=42,
    shuffle = True,
    stratify=y
)

# 1% Validacion - 15% test 
X_val, X_test, y_val, y_test, municipio_val, municipio_test = train_test_split(
    X_temp, y_temp, municipio_temp,
    test_size=0.50,
    random_state=42,
    shuffle = True,
    stratify=y_temp
)

In [44]:
data["nivel_ingles"].value_counts(normalize=True).sort_index()

nivel_ingles
0    0.503763
1    0.496237
Name: proportion, dtype: float64

## Creación de red

In [45]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Validación cruzada
X_dev = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_dev = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)


X_dev = X_dev.astype("float32")
X_test = X_test.astype("float32")
y_dev = y_dev.astype("int32")
y_test = y_test.astype("int32")

num_classes = len(np.unique(y_dev))
input_dim = X_dev.shape[1]

def build_model(input_dim, num_classes, hidden_layers=(64, 32), dropout_rate=0.2, learning_rate=1e-3):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


### Entrenamiento del modelo

In [49]:
param_grid = {
    "hidden_layers": [(16,16), (8,24, 8), (16, 8, 16)],
    "dropout_rate": [0.2],
    "learning_rate": [1e-3],
    "batch_size": [64],
    "epochs": [5]
}

skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
results = []

for params in ParameterGrid(param_grid):
    fold_losses = []
    fold_f1s = []

    print(f"Testing: {params}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
        X_tr = X_dev.iloc[train_idx].copy()
        X_va = X_dev.iloc[val_idx].copy()
        y_tr = y_dev.iloc[train_idx].copy()
        y_va = y_dev.iloc[val_idx].copy()

        # StandardScaler
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)

        # SMOTE en entrenamiento 
        smote = SMOTE(random_state=SEED)
        X_tr_bal, y_tr_bal = smote.fit_resample(X_tr_scaled, y_tr)

        model = build_model(
            input_dim=input_dim,
            num_classes=num_classes,
            hidden_layers=params["hidden_layers"],
            dropout_rate=params["dropout_rate"],
            learning_rate=params["learning_rate"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True
        )

        model.fit(
            X_tr_bal, y_tr_bal,
            validation_data=(X_va_scaled, y_va),
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            verbose=1,
            callbacks=[early_stop]
        )

        y_prob = model.predict(X_va_scaled, verbose=1)
        y_pred = np.argmax(y_prob, axis=1)

        fold_loss = log_loss(y_va, y_prob, labels=np.arange(num_classes))
        fold_f1 = f1_score(y_va, y_pred, average="macro")

        fold_losses.append(fold_loss)
        fold_f1s.append(fold_f1)

        print(f"  Fold {fold}: val_loss={fold_loss:.4f}, f1_macro={fold_f1:.4f}")

    results.append({
        **params,
        "mean_val_loss": np.mean(fold_losses),
        "std_val_loss": np.std(fold_losses),
        "mean_f1_macro": np.mean(fold_f1s),
        "std_f1_macro": np.std(fold_f1s)
    })

results_df = pd.DataFrame(results).sort_values(
    by=["mean_f1_macro", "mean_val_loss"],
    ascending=[False, True]
).reset_index(drop=True)

print(results_df.head())
best_params = results_df.iloc[0].to_dict()
print("\nBest params:")
print(best_params)

Testing: {'batch_size': 64, 'dropout_rate': 0.2, 'epochs': 5, 'hidden_layers': (16, 16), 'learning_rate': 0.001}
Epoch 1/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.6126 - loss: 0.6858 - val_accuracy: 0.6582 - val_loss: 0.6176
Epoch 2/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.6427 - loss: 0.6357 - val_accuracy: 0.6576 - val_loss: 0.6174
Epoch 3/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.6493 - loss: 0.6277 - val_accuracy: 0.6575 - val_loss: 0.6175
Epoch 4/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.6519 - loss: 0.6260 - val_accuracy: 0.6586 - val_loss: 0.6168
Epoch 5/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.6506 - loss: 0.6246 - val_accuracy: 0.6579 - val_loss: 0.6167
1279/1279 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
  Fold 1: val_loss=0.6167, f1_macro=0.6561
Epoch 1/5
643/643 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.6276 - loss: 0.6525 - val_accuracy: 0.6569 - val_loss: 0.6184
Epoch 2/5
643/643 ━━━━━━━━━

### Test modelo

In [48]:

final_scaler = StandardScaler()
X_dev_scaled = final_scaler.fit_transform(X_dev)
X_test_scaled = final_scaler.transform(X_test)

# Balnceo de entrenamiento
smote = SMOTE(random_state=SEED)
X_dev_bal, y_dev_bal = smote.fit_resample(X_dev_scaled, y_dev)

final_model = build_model(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_layers=best_params["hidden_layers"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = final_model.fit(
    X_dev_bal, y_dev_bal,
    validation_split=0.1,
    epochs=int(best_params["epochs"]),
    batch_size=int(best_params["batch_size"]),
    verbose=1,
    callbacks=[early_stop]
)

y_test_prob = final_model.predict(X_test_scaled, verbose=0)
y_test_pred = np.argmax(y_test_prob, axis=1)

test_loss = log_loss(y_test, y_test_prob, labels=np.arange(num_classes))
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")

print(f"Test cross-entropy: {test_loss:.4f}")
print(f"Test F1 macro: {test_f1_macro:.4f}")
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

Epoch 1/2
578/578 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5711 - loss: 0.7192 - val_accuracy: 0.6439 - val_loss: 0.6333
Epoch 2/2
578/578 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6208 - loss: 0.6496 - val_accuracy: 0.6475 - val_loss: 0.6251
Test cross-entropy: 0.6255
Test F1 macro: 0.6524
              precision    recall  f1-score   support

           0       0.64      0.69      0.67      7252
           1       0.66      0.61      0.64      7192

    accuracy                           0.65     14444
   macro avg       0.65      0.65      0.65     14444
weighted avg       0.65      0.65      0.65     14444

[[5028 2224]
 [2788 4404]]
